<a href="https://www.kaggle.com/code/dulapurkaystha/snack-guardian-ai?scriptVersionId=281840802" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Snack Guardian AI: A Multi-Agent Gut-Friendly Snack Assistant

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [2]:
from google.adk.agents import Agent, LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
import json


print("✅ ADK components imported successfully.")

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

MODEL_NAME = "gemini-2.5-flash-lite"
APP_NAME = "snack_concierge_app"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session

✅ ADK components imported successfully.


In [3]:
# Define helper functions that will be reused throughout the notebook
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
):
    print(f"\n ### Session: {session_name}")

    # Get app name from the Runner
    app_name = runner_instance.app_name

    # Attempt to create a new session or retrieve an existing one
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )

    # Process queries if provided
    if user_queries:
        # Convert single query to list for uniform processing
        if type(user_queries) == str:
            user_queries = [user_queries]

        # Process each query in the list sequentially
        for query in user_queries:
            print(f"\nUser > {query}")

            # Convert the query string to the ADK Content format
            query = types.Content(role="user", parts=[types.Part(text=query)])

            # Stream the agent's response asynchronously
            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query
            ):
                # Check if the event contains valid content
                if event.content and event.content.parts:
                    # Filter out empty or "None" responses before printing
                    if (
                        event.content.parts[0].text != "None"
                        and event.content.parts[0].text
                    ):
                        print(f"{MODEL_NAME} > ", event.content.parts[0].text)
    else:
        print("No queries!")


print("✅ Helper functions defined.")

✅ Helper functions defined.


In [13]:
# ======================================
# RAG SYSTEM
# ======================================

KNOWLEDGE_BASE = [
    {
        "condition": "GERD",
        "safe": [
            "oatmeal", "banana", "melon", "ginger tea", "fennel tea",
            "rice", "steamed vegetables", "whole grain toast",
            "sweet potato", "cucumber", "pear"
        ],
        "avoid": [
            "chocolate", "caffeine", "coffee", "mint",
            "citrus", "orange", "tomato", "spicy foods",
            "fried foods", "carbonated drinks", "onions",
            "garlic", "alcohol"
        ]
    },

    {
        "condition": "IBS-C",
        "safe": [
            "kiwi", "prunes", "pear", "chia seeds", "ground flax",
            "oatmeal", "warm herbal teas", "lentil soup",
            "stewed apples", "cooked leafy greens"
        ],
        "avoid": [
            "white bread", "excess cheese", "fried foods",
            "heavy meats", "low-fiber snacks", "low-water meals",
            "unripe bananas"
        ]
    },

    {
        "condition": "IBS-D",
        "safe": [
            "white rice", "oatmeal", "banana", "smooth peanut butter",
            "sweet potatoes", "ripe fruit", "white toast",
            "broth soups", "soluble fiber foods"
        ],
        "avoid": [
            "fried foods", "alcohol", "coffee", "high-fat foods",
            "beans", "lactose-heavy foods", "artificial sweeteners",
            "raw cruciferous vegetables", "large salads"
        ]
    },

    {
        "condition": "Lactose Intolerance",
        "safe": [
            "plant-based yogurt", "almond milk", "soy milk",
            "lactose-free milk", "vegan cheese", "oatmeal",
            "rice dishes", "fruits", "vegetables"
        ],
        "avoid": [
            "milk", "ice cream", "soft cheese",
            "whipped cream", "butter (some tolerate ghee)",
            "milk chocolate", "dairy-heavy snacks"
        ]
    },

    {
        "condition": "Crohn's",
        "safe": [
            "white rice", "oatmeal", "bananas", "smooth nut butters",
            "broth soups", "well-cooked vegetables", "rice noodles",
            "mashed potatoes", "avocado", "ripe fruit"
        ],
        "avoid": [
            "popcorn", "nuts", "seeds", "raw vegetables",
            "corn", "fried foods", "high-fiber cereal",
            "beans", "spicy foods"
        ]
    }
]


def medical_rag_tool(query:str) -> str:
    """
    Consults the medical knowledge base to check if foods are safe or to find soothing foods.
    ARGS:
        query is a string describing the user's condition and/or asking about specific foods.
    
    RETURN:
        A JSON-formatted string representing a list of matched condition entries from the knowledge base. 
        If no match is found, the function returns a simple string:
            "No specific gut dietary data found for this query in the local knowledge base."
    """
    results = []
    q = query.lower()
    for entry in KNOWLEDGE_BASE:
        condition_match = entry["condition"].lower() in q
        safe_match = any(food in q for food in entry["safe"])
        avoid_match = any(food in q for food in entry["avoid"])

        if condition_match or safe_match or avoid_match:
            results.append(entry)

    if not results:
        return "No specific medical dietary data found for this query."
        
    return json.dumps(results, indent=2)

print("Medical RAG Tool complete")
    

Medical RAG Tool complete


In [14]:
root_agent = LlmAgent(
    name="helpful_snack_agent",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config
    ),
    # description="A simple snack agent that can suggest snacks."
    instruction="""
    You are the 'Gut-Friendly Snack Concierge'.
    Your primary goal is to provide snack recommendations based on user's dietary restrictions.
    
    **Your Workflow:**
    1. If the user mentions a gut condition (e.g., "I have GERD", "I have IBS"),
       call `medical_rag_tool` with their full question.
    2. Read the returned JSON and:
       - Explain which foods are generally safer ("safe" list).
       - Explain which foods are better to avoid ("avoid" list).
    3. Suggest a few simple snack ideas based on the safe foods.
    4. Always include a short reminder that this is general comfort guidance,
       not medical advice, and that they should follow their doctor's guidance
    """,
    tools=[medical_rag_tool]
)

db_url = "sqlite:///my_profile_data.db"
session_service = DatabaseSessionService(db_url=db_url)

print(f"   - Database: my_agent_data.db")

print("✅ Root Agent defined.")

   - Database: my_agent_data.db
✅ Root Agent defined.


In [15]:
runner = Runner(
    agent=root_agent, 
    app_name=APP_NAME, 
    session_service=session_service,
)

print("✅ Runner created.")

✅ Runner created.


In [16]:
# response = await runner.run_debug(
#     "I have mild acid reflux. Can you suggest a gentle evening snack?"
# )

# response = await runner.run_debug(
#     "My name is Sally. I have GERD."
# )

# response = await runner.run_debug(
#     "What do you know about my gut triggers now? and What's my name",
# )

await run_session(
    runner, ["Hello! I'm Sally and I'm vegetarian"], "test-sally-01"
)

await run_session(
    runner, ["Hello! Whats my name and what are my diet issues?"], "test-sally-01"
)

await run_session(
    runner, ["I have GERD."], "test-sally-01"
)

await run_session(
    runner, ["Hello! Whats my name and what are my diet issues?"], "test-sally-01"
)


 ### Session: test-sally-01

User > Hello! I'm Sally and I'm vegetarian
gemini-2.5-flash-lite >  Hi Sally! I can help you with snack ideas. Do you have any specific gut conditions I should be aware of (like GERD, IBS, etc.)?

 ### Session: test-sally-01

User > Hello! Whats my name and what are my diet issues?
gemini-2.5-flash-lite >  You are Sally! You've mentioned that you are vegetarian. Do you have any specific gut conditions you're dealing with that I should know about?


 ### Session: test-sally-01

User > I have GERD.


gemini-2.5-flash-lite >  When you have GERD, it's generally recommended to focus on foods that are less likely to trigger acid reflux. Based on the information available:

**Foods generally considered safer:**
*   Oatmeal
*   Banana
*   Melon
*   Ginger tea
*   Fennel tea
*   Rice
*   Steamed vegetables
*   Whole grain toast
*   Sweet potato
*   Cucumber
*   Pear

**Foods generally better to avoid:**
*   Chocolate
*   Caffeine, coffee
*   Mint
*   Citrus (like oranges)
*   Tomatoes
*   Spicy foods
*   Fried foods
*   Carbonated drinks
*   Onions
*   Garlic
*   Alcohol

Here are a few simple snack ideas that might be suitable for you, keeping in mind the "safe" list:

1.  **Oatmeal with Banana:** A warm bowl of oatmeal topped with sliced banana can be very soothing.
2.  **Rice Cakes with Pear:** Plain rice cakes topped with a thin layer of pear slices.
3.  **Steamed Sweet Potato:** A small portion of plain steamed sweet potato.
4.  **Banana:** A simple, ripe banana on its own.

This is 